# 01 · Build the lakehouse (bronze → silver → gold)

Runs in a **Microsoft Fabric** notebook attached to the `LeadScoring` lakehouse.

**Before running:**
1. Upload `data/raw/bank-additional-full.csv` to the lakehouse at `Files/raw/`.
2. Attach the `leadscore` wheel (`python -m build` in the repo, then add `dist/*.whl` to a Fabric Environment), **or** run the `%pip` cell below.

Outputs (Delta tables): `bronze_bank_calls`, `silver_calls`, `gold_lead_features`, `gold_campaign_monthly`.

In [ ]:
# Option B: install the package from the wheel uploaded to Files/dist
# %pip install /lakehouse/default/Files/dist/leadscore-0.1.0-py3-none-any.whl

In [ ]:
from leadscore import lakehouse as lh, pipeline as pl, quality as dq

spark = lh.get_spark()
print('Running in Fabric:', lh.in_fabric())

## Bronze: land the raw file with lineage
The CSV is in date order but has no date or ID, so the original row order is captured as `source_row_number` before anything can reorder it.

In [ ]:
bronze = pl.build_bronze(spark, lh.raw_file_path(pl.SOURCE_FILE))
lh.write_table(bronze, 'bronze_bank_calls')
bronze = lh.read_table(spark, 'bronze_bank_calls')
bronze_rows = bronze.count()
bronze_rows

## Silver: type, clean, de-duplicate, reconstruct call dates
- `pdays = 999` is a sentinel for *never contacted*, so it becomes null plus a flag.
- The call **year** is reconstructed from the month sequence (a new year starts whenever the month number goes backwards). A quality check confirms the result matches the documented May 2008 to Nov 2010 range.
- 12 exact duplicate rows are removed.

In [ ]:
silver = pl.build_silver(bronze)
dq.enforce(dq.check_silver(silver, bronze_rows), 'silver')
lh.write_table(silver, 'silver_calls')
silver = lh.read_table(spark, 'silver_calls')

## Gold: model features and monthly summary
`call_duration_sec` is **excluded** from features: it is only known after the call ends, so using it would leak the outcome.

In [ ]:
gold = pl.build_gold_features(silver)
dq.enforce(dq.check_gold_features(gold, silver.count()), 'gold')
lh.write_table(gold, 'gold_lead_features')
lh.write_table(pl.build_gold_monthly(silver), 'gold_campaign_monthly')

In [ ]:
display(lh.read_table(spark, 'gold_campaign_monthly').orderBy('call_month_start'))